# Reproducing Nakamuta et al. (2025) from dandiset 002020

**Paper.** "Self-organization of high-dimensional geometry of neural activity in culture",
bioRxiv doi:10.1101/2025.01.14.630383. The claims file
(`claims/002020/10.1101-2025.01.14.630383.yaml`) was written from version 1 of the preprint,
flattened from the bioRxiv JATS XML (Results, legends, Discussion, Methods; the supplement and the
equations of the Methods are not in that text).

**Dataset.** [DANDI:002020](https://dandiarchive.org/dandiset/002020/0.261004.1410), version
`0.261004.1410`. The paper cites no version of the dataset. This is the only published version
(published 2026-10-04); the draft has the same 129 assets and size, so the most recent published
version is pinned. It holds 129 NWB files, one 30-minute recording each, from seven dishes
(subjects), with unsorted channel-level spike detections and electrode positions.

**What can be tested.** The files hold recording dates, not days in vitro (DIV): every file says
"Days in vitro are unknown". Claims whose result is a DIV are blocked
(`metadata_insufficient`). Claims that only compare early and late recordings use DIV = days
since the dish's first recording + d0, with d0 = 2 as an assumption (readings d0 = 1 and 3 are
also shown; see "Days in vitro" below). The fitted exponential decay rate b of the variance
spectra is about 0.001 to 0.03 in every active recording, so the paper's 0.1 threshold between
its Exponential and Power-law groups does not separate recordings here; claims that rest on
that threshold for the PCA b are blocked (`methods_underspecified`), with their results shown.

| Claim | Tested |
|---|---|
| dishes-analysed | tested |
| activity-onset | blocked, metadata_insufficient (no DIV) |
| fig1b-async-to-bursts | tested, assumption on d0 |
| fig1d-power-law-emerges | blocked, methods_underspecified (b scale) |
| fig1e-example-dish | blocked, metadata_insufficient (no DIV) |
| fig1f-transition-15div | blocked, metadata_insufficient (no DIV) |
| fig1f-exponent-1p2 | blocked, depends on fig1f-transition-15div |
| fig1d-shuffle | blocked, methods_underspecified (b scale) |
| fig2b-spatial-structure | tested, assumption on d0 |
| fig2c-locality-spectrum | tested, assumption on d0 |
| fig2d-locality-slope | tested |
| fig2d-shuffle-locality | tested, assumption on d0 |
| fig3c-avalanches-emerge | blocked, metadata_insufficient (no DIV) |
| fig3d-decay-rates-correlated | tested |
| fig3ef-groups-cooccur | blocked, methods_underspecified (b scale) |
| figs1-classified-by-avalanches | blocked, methods_underspecified (b scale) |
| figs2-shuffle-breaks-cooccurrence | blocked, methods_underspecified (readings disagree) |
| no-temporal-structure | tested, assumption on d0 |
| fig4d-cauchy-model-power-law | blocked, methods_underspecified (model not specified) |
| fig4d-gaussian-model-no-power-law | blocked, methods_underspecified (model not specified) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import resource
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import culture as C

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 40)
print("dandiset", C.DANDISET_ID, "version", C.DANDISET_VERSION)

## Loading

Every file is streamed and reduced as it loads (`culture.analyse`): 100 ms counts, the
variance spectrum of the original and the shuffled counts, the locality of every PC, cascade size
distributions, synchrony and PC time scales, under the selection readings S1 and S2. Three
processes each hold one file at a time.

In [ ]:
t0 = time.time()
assets = C.list_assets()
R = C.analyse_all(assets, workers=3)
print(f"{len(R)} recordings analysed in {time.time() - t0:.0f} s; peak memory of a worker "
      f"{resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.1f} GB")

## Data check

In [ ]:
meta = pd.DataFrame([dict(dish=r["dish"], date=r["date"], n_electrodes=r["n_el"],
                          n_spikes=r["meta"]["n_spikes_total"], unmapped=r["meta"]["n_spikes_unmapped"],
                          duration_s=r["meta"]["duration_s"], t_max=r["meta"]["t_max"], fs=r["meta"]["fs"],
                          mean_rate=r["mean_rate"], n_above_003=r["n_above"],
                          frames_match_times=r["meta"]["frames_match_times"],
                          sorted=r["meta"]["times_sorted"], grid_ok=r["meta"]["grid_ok"],
                          age=r["meta"]["subject_age"]) for r in R])
meta["day"] = C.day_index(R)
print("recordings per dish and date span:")
print(meta.groupby("dish").agg(n=("date", "size"), first=("date", "min"), last=("date", "max"),
                               span_days=("day", "max")).to_string())
print("\nelectrodes per recording:", meta.n_electrodes.min(), "to", meta.n_electrodes.max(),
      "(the paper: up to 1,024)")
print("recording duration (s):", meta.duration_s.min(), "to", meta.duration_s.max(),
      "(the paper: 30 minutes); last spike at", meta.t_max.min().round(1), "to", meta.t_max.max().round(1),
      "s after the first")
print("sampling rate (Hz):", sorted(meta.fs.unique()))
print("spike times equal (frame - first frame) / 20 kHz in every file:", meta.frames_match_times.all())
print("spike times sorted in every file:", meta.sorted.all())
print("electrode x, y equal the MaxOne grid (17.5 um pitch, 220 columns) of the electrode id:",
      meta.grid_ok.all())
print("spikes on channels with no electrode (dropped):", meta.unmapped.sum(), "of", meta.n_spikes.sum(),
      f"({meta.unmapped.sum() / meta.n_spikes.sum():.2e})")
print("subject age field:", meta.age.unique(), "(the files say it is the donor's E18, not DIV)")
print("mean rate per electrode (Hz): %.4f to %.3f" % (meta.mean_rate.min(), meta.mean_rate.max()))
print("recordings with mean rate <= 0.03 Hz:", (meta.mean_rate <= C.RATE_HZ).sum(), "of", len(meta))
print("recordings with fewer than 10 electrodes above 0.03 Hz:", (meta.n_above_003 < C.MIN_POINTS).sum())
print(meta[["dish", "date", "day", "n_electrodes", "n_spikes", "mean_rate", "n_above_003"]].to_string())

Recordings deposited twice: no two files share a spike count, and recordings of different dishes
made at the same time (dishes 23280 and 23311 on 2024-05-16 to 06-02, 23282 and 23302 on
2024-06-21 to 07-01, which were recorded within a minute of each other) are compared below by the
correlation of their population spike counts in 100 ms bins. The time origin of each file is its
first spike, so a lag search of up to 60 s is made.

In [ ]:
print("files with the same spike count:", meta.n_spikes.duplicated().sum())
by = {(r["dish"], r["date"]): r for r in R}
pairs = [(a, b) for a, b in (("23280", "23311"), ("23282", "23302"), ("23302", "23317"))]
rows = []
for a, b in pairs:
    for (dish, date), r in by.items():
        if dish == a and (b, date) in by:
            p, q = r["pop"].astype(float), by[(b, date)]["pop"].astype(float)
            best = 0.0
            for lag in range(-600, 601, 1):
                if lag >= 0:
                    x, y = p[lag:], q[:len(q) - lag]
                else:
                    x, y = p[:lag], q[-lag:]
                if x.std() > 0 and y.std() > 0:
                    best = max(best, np.corrcoef(x, y)[0, 1])
            rows.append(dict(dishes=f"{a}/{b}", date=date, max_corr=round(best, 3)))
dup = pd.DataFrame(rows)
print(dup.to_string())
print("largest correlation at any lag:", dup.max_corr.max())

**Clock.** Each file holds one data stream, the spike detections, and no trial or annotation
table, so there is nothing to align; spike times run from the first detected spike, as the files
state (the recording-start frame is not stored). Electrode positions are static.

**Days in vitro.** The files give recording dates only. The dates run over 17 to 32 days per dish
with some days missing. The Results say "Spontaneous activity became detectable approximately
2 – 3 days in vitro (DIV)" and that activity "was measured once a day". Five dishes are already
active (mean rate above 0.03 Hz) on their first recording, and two (23302, 23317) are near
silent on it and active on the next one. So the first recording falls at or just before the
onset of activity, which the paper puts at 2 to 3 DIV. Assumption: DIV = day + d0 with d0 = 2
for every dish; d0 = 1 and d0 = 3 are shown too. Claims whose result is itself a DIV cannot rest
on this and are blocked.

In [ ]:
D0 = 2
D0_READINGS = (1, 2, 3)
T = C.fit_table(R)
T["active_S2"] = T.mean_rate > C.RATE_HZ
print("rows (recording x selection x fit):", len(T))
print("recordings with a spectrum under S1:", T[(T.sel == "S1") & (T.F == "F1")].b.notna().sum(),
      "and under S2:", T[(T.sel == "S2") & (T.F == "F1")].b.notna().sum(), "of", len(R))
print("electrodes kept under S1 per recording: median", int(T[T.sel == "S1"].n_kept.median()),
      "range", T[T.sel == "S1"].n_kept.min(), "to", T[T.sel == "S1"].n_kept.max())
print("\nPCA exponential decay rate b, by selection and fit reading (active recordings):")
print(T.dropna(subset=["b"]).groupby(["sel", "F"]).b.describe(percentiles=[.05, .5, .95]).round(4).to_string())
print("\nrecordings with PCA b >= 0.1 (the paper's Exponential group):")
print(T[T.b >= C.B_THRESHOLD][["dish", "day", "sel", "F", "mean_rate", "n_kept", "b", "c"]].round(4).to_string())


def div(day, d0=D0):
    return day + d0


def by_reading(df, extra=()):
    return df.groupby(["sel", "F", *extra])

The values of b (rank as x) are two orders of magnitude below 0.1 in all but a few near-silent
recordings. This is the scale problem said in the header; it is shown per claim below.

## dishes-analysed

> Seven dishes met the paper's inclusion rule (spontaneous activity observed continuously for more
> than 10 days) and were summarized.

Criterion: count the dishes in which recordings with activity above 0.03 Hz (S1, at least one
electrode above 0.03 Hz; S2, mean rate above 0.03 Hz) run continuously, without a day missing
activity, over more than 10 days. Whether a day without a recording breaks the run is not said;
both are shown: (i) a missing date breaks it, (ii) only a recorded day without activity breaks it.

In [ ]:
def longest_run(days, active, missing_breaks):
    best, start, prev = 0, None, None
    for d, a in zip(days, active):
        if not a:
            start, prev = None, None
            continue
        if start is None or (missing_breaks and d != prev + 1):
            start = d
        prev = d
        best = max(best, d - start + 1)
    return best


rows = []
for dish, g in meta.sort_values(["dish", "day"]).groupby("dish"):
    for sel, act in (("S1", g.n_above_003 > 0), ("S2", g.mean_rate > C.RATE_HZ)):
        for rule, mb in (("i, missing date breaks", True), ("ii, only inactive day breaks", False)):
            rows.append(dict(dish=dish, sel=sel, rule=rule, longest_run_days=longest_run(g.day.values, act.values, mb)))
runs = pd.DataFrame(rows)
print(runs.pivot_table(index="dish", columns=["sel", "rule"], values="longest_run_days").to_string())
counts = runs.assign(ok=runs.longest_run_days > 10).groupby(["sel", "rule"]).ok.sum()
print("\ndishes with a run of more than 10 days:")
print(counts.to_string())
print("dishes in the dataset:", meta.dish.nunique())

**Result: reproduced, resting on an assumption.** Under S1, with only a recorded day without
activity breaking a run (reading ii, assumed: a day with no recording is not a day without
activity), all 7 dishes have a run of more than 10 days (11 to 33 days). If a day with no
recording also breaks the run (reading i), 6 dishes do (23317 has 10 days); under S2, 2 or 4 do.
The dataset holds exactly 7 dishes.

## activity-onset (blocked)

> Spontaneous activity became detectable at about 2 to 3 DIV.

Blocked, `metadata_insufficient`: the files give no DIV and no plating date. Shown for the record:
the first day (since the dish's first recording) with activity above 0.03 Hz.

In [ ]:
first_active = []
for dish, g in meta.sort_values("day").groupby("dish"):
    s1 = g[g.n_above_003 > 0].day.min()
    s2 = g[g.mean_rate > C.RATE_HZ].day.min()
    first_active.append(dict(dish=dish, first_day_S1=s1, first_day_S2=s2,
                             rate_first_recording=round(g.mean_rate.iloc[0], 4)))
print(pd.DataFrame(first_active).to_string())

## fig1b-async-to-bursts

> As the cultures matured, spiking changed from asynchronous firing to synchronous bursting.

Criterion: (a) mean pairwise Pearson correlation of the 100 ms counts, (b) the largest fraction
of electrodes active within one 100 ms bin; over all recordings against DIV, Spearman positive with
p < 0.05, and the dish median at 15 DIV and later above that at 3 to 5 DIV. Readings S1 and S2;
DIV = day + d0 (assumption, d0 = 1, 2, 3). "Dish median" is read as the median over dishes of each
dish's median in the window; the median over recordings is shown too.

In [ ]:
def window_compare(df, col, d0, early=(3, 5), late=15):
    dv = div(df.day, d0)
    e = df[(dv >= early[0]) & (dv <= early[1])]
    l = df[dv >= late]
    de = e.groupby("dish")[col].median()
    dl = l.groupby("dish")[col].median()
    return dict(early_dish_median=de.median(), late_dish_median=dl.median(),
                early_rec_median=e[col].median(), late_rec_median=l[col].median(),
                n_early=len(e), n_late=len(l), dishes_early=len(de), dishes_late=len(dl))


rows = []
F1 = T[(T.F == "F1")].dropna(subset=["b"])
for sel, g in F1.groupby("sel"):
    for col in ("sync_corr", "sync_frac"):
        rho, p = stats.spearmanr(g.day, g[col])
        for d0 in D0_READINGS:
            w = window_compare(g, col, d0)
            rows.append(dict(sel=sel, measure=col, d0=d0, spearman=rho, p=p, **w,
                             holds=(rho > 0) and (p < 0.05) and (w["late_dish_median"] > w["early_dish_median"])))
fig1b = pd.DataFrame(rows)
print(fig1b.round(4).to_string())
print("\nper dish Spearman of synchrony with day (S1):")
print(F1[F1.sel == "S1"].groupby("dish").apply(
    lambda g: pd.Series({m: stats.spearmanr(g.day, g[m])[0] for m in ("sync_corr", "sync_frac")})).round(3).to_string())
print("\nrange of sync_corr", F1.sync_corr.min().round(4), F1.sync_corr.max().round(4),
      "; sync_frac", F1.sync_frac.min().round(4), F1.sync_frac.max().round(4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for dish, g in F1[F1.sel == "S1"].groupby("dish"):
    ax[0].plot(div(g.day), g.sync_corr, "o-", ms=3, label=dish)
    ax[1].plot(div(g.day), g.sync_frac, "o-", ms=3)
ax[0].set(xlabel="DIV (day + 2, assumed)", ylabel="mean pairwise correlation")
ax[1].set(xlabel="DIV (day + 2, assumed)", ylabel="largest fraction active in a bin")
ax[0].legend(fontsize=7)
plt.tight_layout()

**Result: reproduced, resting on the assumption d0 = 2.** Over all recordings the mean pairwise
correlation rises with day (Spearman 0.487 under S1, 0.384 under S2, p < 0.0001), and so does the
largest fraction of electrodes active in one bin (0.301, p 0.0007; 0.253, p 0.0072). With d0 = 2
and d0 = 3 the dish median from 15 DIV exceeds that at 3 to 5 DIV for both measures and both
selections (correlation 0.330 against 0.063 under S1, d0 = 2). With d0 = 1 the window comparison
fails for the fraction active under S1 and for both measures under S2. Not every dish shows the
rise: in 23282 both measures fall with day (Spearman -0.227 and -0.273).

## fig1d-power-law-emerges (blocked)

> In early cultures the variance spectrum decays faster than a power function; as the culture
> matures it follows a power function, linear on log-log axes.

Criterion: in a majority of the seven dishes, the recordings of the first three recorded DIV have
b >= 0.1 and those from 15 DIV on have b < 0.1 (F1 and F2). Computed as written, and with the
threshold-free trend of b and c against day.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for d0 in D0_READINGS:
        ok = 0
        for dish, h in g.groupby("dish"):
            first3 = h[h.day.isin(sorted(h.day.unique())[:3])]
            late = h[div(h.day, d0) >= 15]
            ok += bool((first3.b >= 0.1).all() and len(late) and (late.b < 0.1).all())
        rows.append(dict(sel=sel, F=F, d0=d0, dishes_meeting=ok,
                         first3_b_ge_01=(g.groupby("dish").apply(lambda h: (h[h.day.isin(sorted(h.day.unique())[:3])].b >= 0.1).sum())).sum()))
print(pd.DataFrame(rows).to_string())
print("\nb and c in the first three recordings and from 15 DIV on (d0 = 2), median over recordings:")
g = T.dropna(subset=["b"]).copy()
g["rank"] = g.groupby(["dish", "sel", "F"]).day.rank(method="dense")
g["phase"] = np.where(g["rank"] <= 3, "first3", np.where(div(g.day) >= 15, "from15", "between"))
print(g.groupby(["sel", "F", "phase"])[["b", "c"]].median().round(4).to_string())
for (sel, F), h in by_reading(T.dropna(subset=["b"])):
    print(sel, F, "Spearman of b with day: rho %.3f p %.2g; of c with day: rho %.3f p %.2g" % (
        *stats.spearmanr(h.day, h.b), *stats.spearmanr(h.day, h.c)))

In [ ]:
# Variants of the fit range and of x, shown for the question to the authors (not part of the criterion).
rows = []
for r, d in zip(R, C.day_index(R)):
    ev = r["S1"].get("ev")
    if ev is None:
        continue
    n = len(ev)
    rows.append(dict(dish=r["dish"], day=d, first3=d <= sorted({dd for rr, dd in zip(R, C.day_index(R)) if rr["dish"] == r["dish"]})[2],
                     b_all=C.fit_decay(ev, "F1")[1], b_top100=C.fit_decay(ev[:100], "F1")[1],
                     b_top30=C.fit_decay(ev[:30], "F1")[1],
                     b_x_over_n=C.fit_decay(ev, "F1", np.arange(1, n + 1) / n)[1]))
var = pd.DataFrame(rows)
print("S1, F1: median b in the first three recordings of each dish and in the others, and the share >= 0.1")
print(var.groupby("first3")[["b_all", "b_top100", "b_top30", "b_x_over_n"]].agg(["median", lambda v: (v >= 0.1).mean()]).round(4).to_string())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ex = [r for r in R if r["dish"] == "19880"]
days_ex = [d for r, d in zip(R, C.day_index(R)) if r["dish"] == "19880"]
cm = plt.cm.viridis
for r, d in zip(ex, days_ex):
    ev = r["S1"].get("ev")
    if ev is not None:
        ax[0].loglog(np.arange(1, len(ev) + 1), ev / ev[0], color=cm(d / max(days_ex)), lw=0.8)
        ax[1].loglog(np.arange(1, len(ev) + 1), r["S1"]["ev_shuf"] / r["S1"]["ev_shuf"][0],
                     color=cm(d / max(days_ex)), lw=0.8)
ax[0].set(title="dish 19880, S1, colour = day", xlabel="PC rank", ylabel="variance / first")
ax[1].set(title="same, shuffled", xlabel="PC rank")
for (sel, F), h in by_reading(T[(T.sel == "S1")].dropna(subset=["b"])):
    ax[2].semilogy(div(h.day), h.b.clip(lower=1e-5), "o", ms=3, label=F, alpha=0.6)
ax[2].axhline(0.1, color="k", ls="--")
ax[2].set(xlabel="DIV (day + 2, assumed)", ylabel="b (clipped at 1e-5)", title="PCA b, S1")
ax[2].legend()
plt.tight_layout()
print("b values below 1e-5 (clipped in the plot):", (T.b < 1e-5).sum(), "of", T.b.notna().sum())

## fig1e-example-dish, fig1f-transition-15div, fig1f-exponent-1p2 (blocked)

Blocked: the shift DIV and the transition DIV need days in vitro, which the files do not give, and
the 0.1 boundary on b is met from the first active recording in every dish (above). The exponent
claim depends on the transition. Shown for the record: per dish, the first day from which b stays
below 0.1, and the median c over recordings from that day (F1, F2).

In [ ]:
rows = []
for (sel, F, dish), h in by_reading(T.dropna(subset=["b"]), ["dish"]):
    h = h.sort_values("day")
    above = np.flatnonzero((h.b >= 0.1).values)
    start = h.day.values[above[-1] + 1] if len(above) and above[-1] + 1 < len(h) else (h.day.values[0] if not len(above) else np.nan)
    after = h[h.day >= start]
    rows.append(dict(sel=sel, F=F, dish=dish, first_day_b_stays_below=start, median_c_after=after.c.median(),
                     last_c=h.c.values[-1]))
tr = pd.DataFrame(rows)
print(tr.round(3).to_string())
for (sel, F), h in by_reading(T.dropna(subset=["b"])):
    print(sel, F, "Spearman of b with day over all recordings: rho %.3f, p %.2g" % stats.spearmanr(h.day, h.b))
print("median c over all active recordings from the transition on:")
print(tr.groupby(["sel", "F"]).median_c_after.median().round(3).to_string())

## fig1d-shuffle (blocked)

> Temporal shuffling leaves the variance spectrum unchanged early in development but, once the
> power law has emerged, disrupts it and returns it to exponential decay.

Criterion: early recordings are the Exponential group (b >= 0.1), late ones the Power-law group;
(i) in early recordings the median ratio of shuffled to original b within 0.9 to 1.1; (ii) in late
recordings shuffled b >= 0.1 in a majority and above the original b, one-sided Wilcoxon p < 0.05.
As written, and with early and late taken as 3 to 5 DIV and from 15 DIV (d0 = 2), a variant.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b", "b_shuf"])):
    for name, early, late in (("groups by b", g[g.b >= 0.1], g[g.b < 0.1]),
                              ("DIV 3-5 / 15+ (d0=2)", g[div(g.day).between(3, 5)], g[div(g.day) >= 15])):
        ratio = (early.b_shuf / early.b).median() if len(early) else np.nan
        p = stats.wilcoxon(late.b_shuf, late.b, alternative="greater").pvalue if len(late) > 1 else np.nan
        rows.append(dict(sel=sel, F=F, split=name, n_early=len(early), early_median_ratio=ratio,
                         n_late=len(late), late_frac_shuf_ge_01=(late.b_shuf >= 0.1).mean(),
                         late_median_b=late.b.median(), late_median_b_shuf=late.b_shuf.median(),
                         late_median_c=late.c.median(), late_median_c_shuf=late.c_shuf.median(), wilcoxon_p=p))
print(pd.DataFrame(rows).round(4).to_string())

## fig2c-locality-spectrum

> The locality spectrum (the number of electrodes with local Moran's I above threshold, against PC
> index) is flat early in development and becomes steeper as the culture matures.

Criterion: under M1 and M2 and L1 and L2, the Spearman correlation of the locality slope with DIV,
over all recordings, negative with p < 0.05, and the median slope at 3 to 5 DIV closer to zero than
that at 15 DIV and later. Slopes are from the original PCs; w_ii = 0 (w_ii = 1 shown as a variant).
Readings S1 and S2; d0 = 1, 2, 3 for the windows (the Spearman correlation does not depend on d0).

In [ ]:
rows = []
L = T[T.F == "F1"].dropna(subset=["b"])
for sel, g in L.groupby("sel"):
    for key in ("M1", "M2", "M1_self", "M2_self"):
        for Lr in ("L1", "L2"):
            col = f"loc_{key}_{Lr}"
            h = g.dropna(subset=[col])
            rho, p = stats.spearmanr(h.day, h[col])
            for d0 in D0_READINGS:
                dv = div(h.day, d0)
                me, ml = h[dv.between(3, 5)][col].median(), h[dv >= 15][col].median()
                rows.append(dict(sel=sel, moran=key, slope=Lr, d0=d0, n=len(h), spearman=rho, p=p,
                                 median_3_5=me, median_15=ml,
                                 holds=(rho < 0) and (p < 0.05) and (abs(me) < abs(ml))))
fig2c = pd.DataFrame(rows)
print(fig2c.to_string(float_format=lambda v: f"{v:.4g}"))
main = fig2c[~fig2c.moran.str.contains("self")]
print("\nmain readings (w_ii = 0) holding:", main.holds.sum(), "of", len(main))

Magnitude of local Moran's I under M1 and M2, on one recording (dish 23312, its last day), to see
what the threshold 0.0001 does under each reading.

In [ ]:
a = [x for x in assets if x["dish"] == "23312"][-1]
t_, e_, x_, y_, _ = C.load_recording(a["asset_id"])
X_ = C.bin_counts(t_, e_, len(x_))
rates_ = X_.sum(axis=1) / C.DURATION_S
keep_ = rates_ > C.RATE_HZ
ev_, V_, _ = C.pca(X_[keep_])
W_ = C.neighbour_matrix(x_[keep_], y_[keep_])
Z_ = V_ - V_.mean(axis=0)
I2 = Z_ * (W_ @ Z_)
I1 = I2 / ((Z_ ** 2).mean(axis=0))
for name, I in (("M1", I1), ("M2", I2)):
    print(f"{a['path']}: {name} median |I| {np.median(np.abs(I)):.3g}; share of values > 0 {np.mean(I > 0):.3f}; "
          f"share > 0.0001 {np.mean(I > C.MORAN_THRESHOLD):.3f}")
del X_, V_, Z_, I1, I2

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for r, d in zip(R, C.day_index(R)):
    if r["dish"] != "23312" or "ev" not in r["S1"]:
        continue
    loc = r["S1"]["loc_M2"]
    sm = np.convolve(loc, np.ones(10) / 10, mode="valid")
    ax[0].semilogx(np.arange(1, len(sm) + 1), sm, color=plt.cm.viridis(d / 25), lw=0.8)
ax[0].set(title="dish 23312, S1, M2, 10-PC moving average", xlabel="PC index", ylabel="locality")
h = L[L.sel == "S1"]
ax[1].plot(div(h.day), h["loc_M2_L1"], "o", ms=3)
ax[1].set(xlabel="DIV (day + 2, assumed)", ylabel="locality slope (M2, L1)")
plt.tight_layout()

**Result: reproduced under the assumed reading M2 with L1; not under M1 or L2.** Under M2 with
L1 (w_ii = 0) the slope falls with day (Spearman -0.318, p 0.00036, S1; -0.401, p 1.2e-05, S2) and
the median slope at 3 to 5 DIV is closer to zero than from 15 DIV for d0 = 1, 2 and 3. The early
spectra are not flat, though: with d0 = 2 the early median is -104.2 against -105.7 late (S1), and
only with d0 = 3 is the early median much nearer zero (-23.95). Under M1, and under L2, the Spearman
correlation is not negative with p < 0.05, so 6 of the 24 main readings hold. Under M1 the
threshold 0.0001 keeps nearly every positive value (share above it 0.394, share positive 0.409),
while under M2 it keeps 0.057, which is why M2 is the reading assumed.

## fig2b-spatial-structure

> Early in development (3 DIV) all PCs correspond to a few scattered electrodes; in mature cultures
> (15 DIV) dominant PCs correspond to localized clusters of electrodes, while minor PCs remain on
> globally distributed, sporadic electrodes.

Criterion: fig2c-locality-spectrum reproduced and, in a majority of dishes at 15 DIV (or the nearest
recorded DIV), the mean locality of the first 10 PCs above that of the last 10, while at 3 DIV it
does not exceed it by more than 10%. M1 and M2 (w_ii = 0); S1 and S2; d0 = 1, 2, 3.

In [ ]:
days = C.day_index(R)
rows = []
for sel in ("S1", "S2"):
    for key in ("M1", "M2"):
        for d0 in D0_READINGS:
            late_ok = early_ok = n_dish = 0
            detail = []
            for dish in sorted({r["dish"] for r in R}):
                recs = [(d, r) for d, r in zip(days, R) if r["dish"] == dish and f"loc_{key}" in r[sel]]
                if not recs:
                    continue
                n_dish += 1
                pick = {}
                for target in (3, 15):
                    d, r = min(recs, key=lambda dr: (abs(div(dr[0], d0) - target), dr[0]))
                    loc = r[sel][f"loc_{key}"]
                    pick[target] = (div(d, d0), loc[:10].mean(), loc[-10:].mean())
                late_ok += pick[15][1] > pick[15][2]
                early_ok += pick[3][1] <= 1.1 * pick[3][2]
                detail.append((dish, *[round(v, 1) for v in pick[3]], *[round(v, 1) for v in pick[15]]))
            rows.append(dict(sel=sel, moran=key, d0=d0, dishes=n_dish, late_first10_gt_last10=late_ok,
                             early_within_10pct=early_ok))
            if sel == "S1" and d0 == D0:
                print(sel, key, "per dish (DIV, first 10, last 10) at about 3 DIV, then at about 15 DIV:")
                for row in detail:
                    print("  ", row)
print(pd.DataFrame(rows).to_string())

**Result: not reproduced.** Under the assumed M2 (as in fig2c-locality-spectrum), at about 15 DIV
the first 10 PCs are more local than the last 10 in 7 of 7 dishes, but at about 3 DIV they already
exceed the last 10 by more than 10% in 6 of 7 dishes (S1, d0 = 2), and the early part fails for
every selection and d0. Of all readings only S2 with M1 and d0 = 3 meets the criterion (late 5 of
7, early 7 of 7).

## fig2d-locality-slope

> The slope of the locality spectrum confirms that spatial structure emerges progressively and is
> absent in the initial stages of development.

Criterion: on the 3-day moving average of the slope over DIV (pooled over dishes; a centred
window of DIV d-1 to d+1, a standard reading of "3-day moving average"), near zero at the first
recorded DIV (magnitude below 10% of its most negative value) and falling over DIV, without
returning to zero once below half of its most negative value. M1, M2, L1, L2; S1 and S2. With one
d0 for all dishes the moving average is the same curve shifted, so d0 does not change the outcome.

In [ ]:
def moving_average(df, col):
    days_u = np.sort(df.day.unique())
    return pd.Series([df[(df.day >= d - 1) & (df.day <= d + 1)][col].mean() for d in days_u], index=days_u)


rows = []
mas = {}
for sel, g in L.groupby("sel"):
    for key in ("M1", "M2"):
        for Lr in ("L1", "L2"):
            col = f"loc_{key}_{Lr}"
            ma = moving_average(g.dropna(subset=[col]), col)
            mas[(sel, key, Lr)] = ma
            mn = ma.min()
            first = ma.iloc[0]
            below_half = np.flatnonzero(ma.values <= 0.5 * mn)
            after = ma.values[below_half[0]:] if len(below_half) else np.array([])
            returns = bool(len(after) and (np.abs(after) < 0.1 * abs(mn)).any())
            rho, p = stats.spearmanr(ma.index, ma.values)
            rows.append(dict(sel=sel, moran=key, slope=Lr, first_ma=first, most_negative=mn,
                             first_over_min=first / mn if mn < 0 else np.nan,
                             day_most_negative=ma.idxmin(), ma_trend_rho=rho, ma_trend_p=p,
                             returns_to_zero=returns,
                             holds=(mn < 0) and abs(first) < 0.1 * abs(mn) and rho < 0 and not returns))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.4g}"))
print("\nmoving average, S1, M2, L1, by day:")
print(mas[("S1", "M2", "L1")].round(1).to_string())

**Result: partially reproduced, under the assumed M2 with L1.** The moving average of the slope
falls over days (from -34.5 on the first day to -186.1 on day 24, Spearman with day -0.729,
p 1.5e-06, S1; -0.771 under S2) and does not return to zero, but on the first day it is already
19% of its most negative value (S2, 19%), above the 10% bound for "absent". Under M1 the moving
average does not fall significantly (Spearman -0.253, p 0.16, S1), and under M2 with L2 it does
not fall (Spearman 0.423 under S1, 0.098 under S2).

## fig2d-shuffle-locality

> Temporal shuffling, which removes inter-electrode correlations, disrupts the spatial structure of
> PCs; the locality spectrum becomes nearly independent of PC index.

Criterion: data with established structure read as (a) recordings from 15 DIV on (d0 = 1, 2, 3),
(b) recordings in the PCA Power-law group (b < 0.1; under F1 and F2 this is nearly every active
recording). Under each of M1, M2, L1, L2, the median shuffled slope below 10% of the median
original slope in magnitude, and the shuffled slopes less negative than the original by a one-sided
Wilcoxon signed-rank test, p < 0.05.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    sets = [(f"a, DIV >= 15, d0={d0}", g[div(g.day, d0) >= 15]) for d0 in D0_READINGS]
    sets.append(("b, Power-law group", g[g.b < 0.1]))
    if F == "F2":
        sets = sets[-1:]
    for name, h in sets:
        for key in ("M1", "M2"):
            for Lr in ("L1", "L2"):
                o, s = h[f"loc_{key}_{Lr}"], h[f"locshuf_{key}_{Lr}"]
                ok = o.notna() & s.notna()
                o, s = o[ok], s[ok]
                p = stats.wilcoxon(s, o, alternative="greater").pvalue if len(o) > 1 else np.nan
                rows.append(dict(sel=sel, F=F, set=name, moran=key, slope=Lr, n=len(o),
                                 median_orig=o.median(), median_shuf=s.median(),
                                 ratio=abs(s.median()) / abs(o.median()), wilcoxon_p=p,
                                 holds=abs(s.median()) < 0.1 * abs(o.median()) and p < 0.05))
fig2ds = pd.DataFrame(rows)
print(fig2ds.to_string(float_format=lambda v: f"{v:.4g}"))
print("\nreadings holding:", fig2ds.holds.sum(), "of", len(fig2ds))
print(fig2ds.groupby(["set", "moran", "slope"]).holds.all().to_string())

**Result: partially reproduced.** Shuffling makes the slopes less negative under every reading
and set (one-sided Wilcoxon p at most 2.2e-07). Under M2 with L1 the shuffled median is below 10%
of the original in every set, for S1 and S2 (ratio 0.001 to 0.094), but under M2 with L2 it is not
in any set (0.12 to 0.73), so no set meets the bound under each of M1, M2, L1 and L2, as the
criterion requires.

## fig3c-avalanches-emerge (blocked)

> Across seven dishes the cascade size distribution shifted from exponential to power-law decay at
> about 15 DIV, with a power exponent of about 1.5.

Blocked, `metadata_insufficient`: the shift DIV needs days in vitro. Shown for the record: per
dish, the first day from which the avalanche b stays below 0.1, and the median c after it
(A1, A2; F1, F2; S1).

In [ ]:
rows = []
for (sel, F, dish), h in by_reading(T[T.sel == "S1"].dropna(subset=["avA1_b"]), ["dish"]):
    h = h.sort_values("day")
    for A in ("A1", "A2"):
        bb = h[f"av{A}_b"].values
        above = np.flatnonzero(bb >= 0.1)
        start = h.day.values[above[-1] + 1] if len(above) and above[-1] + 1 < len(h) else (h.day.values[0] if not len(above) else np.nan)
        rows.append(dict(F=F, A=A, dish=dish, first_day_stays_below=start,
                         median_c_after=h[h.day >= start][f"av{A}_c"].median()))
av = pd.DataFrame(rows)
print(av.pivot_table(index="dish", columns=["F", "A"], values=["first_day_stays_below", "median_c_after"]).round(2).to_string())
print("\nmedian over dishes:")
print(av.groupby(["F", "A"])[["first_day_stays_below", "median_c_after"]].median().round(2).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
sizes = np.arange(1, 129)
for r, d in zip(R, C.day_index(R)):
    if r["dish"] == "23312" and r["S1"].get("aval"):
        ax[0].loglog(sizes, r["S1"]["aval"]["A1"], color=plt.cm.viridis(d / 25), lw=0.8)
ax[0].set(title="dish 23312, S1, A1, colour = day", xlabel="cascade size S", ylabel="P(S)")
h = T[(T.sel == "S1") & (T.F == "F1")]
ax[1].plot(div(h.day), h.avA1_b, "o", ms=3)
ax[1].axhline(0.1, color="k", ls="--")
ax[1].set(xlabel="DIV (day + 2, assumed)", ylabel="avalanche b (A1, F1)")
plt.tight_layout()

## fig3d-decay-rates-correlated

> The exponential decay rates of the variance spectrum and of the cascade size distribution are
> positively correlated across recordings, so the two power laws are established concurrently.

Criterion: one point per recording, PCA b against avalanche b, under F1, F2, A1, A2 (and S1, S2);
Spearman and Pearson both positive with two-sided p < 0.05. Per-dish values are reported.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        rs, ps = stats.spearmanr(h.b, h[f"av{A}_b"])
        rp, pp = stats.pearsonr(h.b, h[f"av{A}_b"])
        per = h.groupby("dish").apply(lambda q: stats.spearmanr(q.b, q[f"av{A}_b"])[0] if len(q) > 2 else np.nan)
        rows.append(dict(sel=sel, F=F, A=A, n=len(h), spearman=rs, p_spearman=ps, pearson=rp, p_pearson=pp,
                         dishes_positive=f"{(per > 0).sum()} of {per.notna().sum()}",
                         holds=rs > 0 and ps < 0.05 and rp > 0 and pp < 0.05))
fig3d = pd.DataFrame(rows)
print(fig3d.to_string(float_format=lambda v: f"{v:.3g}"))
print("\nwithout the recordings at or below 0.03 Hz mean rate (S1 keeps them when some electrodes pass):")
for (sel, F), g in by_reading(T[T.b.notna() & (T.mean_rate > C.RATE_HZ)]):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        print(sel, F, A, "n %d Spearman %.3f (p %.2g) Pearson %.3f (p %.2g)" % (
            len(h), *stats.spearmanr(h.b, h[f"av{A}_b"]), *stats.pearsonr(h.b, h[f"av{A}_b"])))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for i, F in enumerate(("F1", "F2")):
    h = T[(T.sel == "S1") & (T.F == F)].dropna(subset=["b"])
    ax[i].scatter(h.b, h.avA1_b, s=8)
    ax[i].set(xlabel="PCA b", ylabel="avalanche b (A1)", title=f"S1, {F}", xscale="symlog", xlim=(0, None))
    ax[i].xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
plt.tight_layout()
print("PCA b range (S1):", T[T.sel == "S1"].b.min(), T[T.sel == "S1"].b.max())

**Result: not reproduced.** Over 112 to 122 recordings the PCA b and the avalanche b do not have
both a positive Spearman and a positive Pearson correlation at p < 0.05 under any of the 8 readings.
The Spearman correlation is 0.025 to 0.217 (p 0.022 only for S2, F1, A1, with Pearson -0.029), and
the Pearson correlation is significant only for S1, F1, A2 (0.484, Spearman 0.066, p 0.47). Per
dish, 3 or 4 of 7 dishes have a positive Spearman correlation.

## fig3ef-groups-cooccur and figs1-classified-by-avalanches (blocked)

fig3ef: groups by PCA b with the threshold 0.1; (a) avalanche b lower in the Power-law group
(one-sided Mann-Whitney U, p < 0.05) and its median below 0.1 in the Power-law group and at or above
0.1 in the Exponential group; (b) the cascade size distribution pooled over each group, fitted, b
below 0.1 for the Power-law group and at or above 0.1 for the Exponential group. figs1: the same with
the roles swapped (groups by avalanche b at 0.1). The group sizes show why these cannot be judged;
a median split of the PCA b is shown as a variant, not as a test of the claim.

In [ ]:
def pooled_b(results_sel, F, A):
    if not results_sel:
        return np.nan
    tot = np.zeros(C.AVALANCHE_MAX)
    n = 0
    for av_ in results_sel:
        tot += av_[A] * av_["n_cascades"]
        n += av_["n_cascades"]
    return C.fit_decay(tot / n, F, np.arange(1, C.AVALANCHE_MAX + 1.0))[1]


key_of = {(r["dish"], r["date"]): r for r in R}
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        for split, thr in (("b < 0.1", 0.1), ("median split (variant)", g.b.median())):
            pw, ex_ = g[g.b < thr], g[g.b >= thr]
            p = stats.mannwhitneyu(pw[f"av{A}_b"].dropna(), ex_[f"av{A}_b"].dropna(), alternative="less").pvalue \
                if len(pw) and len(ex_) else np.nan
            pb = [pooled_b([key_of[(d, t)][sel]["aval"] for d, t in zip(grp.dish, grp.date)], F, A) for grp in (pw, ex_)]
            rows.append(dict(sel=sel, F=F, A=A, split=split, n_power=len(pw), n_exp=len(ex_),
                             aval_b_median_power=pw[f"av{A}_b"].median(), aval_b_median_exp=ex_[f"av{A}_b"].median(),
                             mwu_p=p, pooled_b_power=pb[0], pooled_b_exp=pb[1]))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b"])
        pw, ex_ = h[h[f"av{A}_b"] < 0.1], h[h[f"av{A}_b"] >= 0.1]
        p = stats.mannwhitneyu(pw.b, ex_.b, alternative="less").pvalue if len(pw) and len(ex_) else np.nan
        rows.append(dict(sel=sel, F=F, A=A, n_power=len(pw), n_exp=len(ex_), pca_b_median_power=pw.b.median(),
                         pca_b_median_exp=ex_.b.median(), mwu_p=p,
                         pca_c_median_power=pw.c.median(), pca_c_median_exp=ex_.c.median()))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))

## figs2-shuffle-breaks-cooccurrence (blocked)

Readings on the shuffled data (each electrode's spike train shifted by its bin shift): (a) in a
majority of Power-law group recordings the shuffled data keep avalanche b below 0.1 while their PCA
b is at or above 0.1; (b) over all recordings the fig3d correlation on shuffled data is not positive
with p < 0.05.

In [ ]:
rows = []
for (sel, F), g in by_reading(T.dropna(subset=["b_shuf"])):
    for A in ("A1", "A2"):
        h = g.dropna(subset=[f"av{A}_b_shuf"])
        pw = h[h.b < 0.1]
        a_frac = ((pw[f"av{A}_b_shuf"] < 0.1) & (pw.b_shuf >= 0.1)).mean()
        rs, ps = stats.spearmanr(h.b_shuf, h[f"av{A}_b_shuf"])
        rp, pp = stats.pearsonr(h.b_shuf, h[f"av{A}_b_shuf"])
        rows.append(dict(sel=sel, F=F, A=A, n_power=len(pw), a_fraction=a_frac,
                         shuf_aval_b_median=pw[f"av{A}_b_shuf"].median(), shuf_pca_b_median=pw.b_shuf.median(),
                         spearman=rs, p_spearman=ps, pearson=rp, p_pearson=pp,
                         a_holds=a_frac > 0.5, b_holds=not ((rs > 0 and ps < 0.05) or (rp > 0 and pp < 0.05))))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))

## no-temporal-structure

> The cultures did not show the in vivo temporal structure of PCs, in which dominant PCs have
> longer time scales than minor PCs.

Criterion: on recordings from 15 DIV on, time scale of each PC's projection as (a) the first lag at
which its autocorrelation falls below 1/e, (b) the integral of its positive part (here, to the
first non-positive lag); the claim holds when the Spearman correlation of time scale with PC rank is
not negative with p < 0.05 in a majority of recordings. S1 and S2; d0 = 1, 2, 3.

In [ ]:
rows = []
per_rec = []
for r, d in zip(R, C.day_index(R)):
    for sel in ("S1", "S2"):
        Q = r[sel]
        if "tau_e" not in Q:
            continue
        rank = np.arange(1, len(Q["tau_e"]) + 1)
        for name in ("tau_e", "tau_i"):
            rho, p = stats.spearmanr(rank, Q[name])
            per_rec.append(dict(dish=r["dish"], day=d, sel=sel, measure=name, rho=rho, p=p,
                                neg_sig=(rho < 0) and (p < 0.05), top10=np.median(Q[name][:10]),
                                bottom10=np.median(Q[name][-10:])))
PR = pd.DataFrame(per_rec)
for (sel, name), g in PR.groupby(["sel", "measure"]):
    for d0 in D0_READINGS:
        h = g[div(g.day, d0) >= 15]
        rows.append(dict(sel=sel, measure=name, d0=d0, n=len(h), n_neg_sig=int(h.neg_sig.sum()),
                         frac_not_neg_sig=1 - h.neg_sig.mean(), median_rho=h.rho.median(),
                         median_top10_s=h.top10.median(), median_bottom10_s=h.bottom10.median(),
                         holds=(1 - h.neg_sig.mean()) > 0.5))
print(pd.DataFrame(rows).to_string(float_format=lambda v: f"{v:.3g}"))
print("recording and measure pairs with a constant time scale (no correlation defined, counted as not negative):",
      int(PR.rho.isna().sum()), "of", len(PR))
print("\nall recordings, fraction with a significant negative correlation:")
print(PR.groupby(["sel", "measure"]).neg_sig.mean().round(3).to_string())

**Result: not reproduced, resting on the assumption on d0.** From 15 DIV on (d0 = 2), the time
scale falls with PC rank, Spearman negative with p < 0.05, in 25 of 44 recordings under (a) and 27
of 44 under (b) (S1; under S2 23 and 31 of 41), so dominant PCs have longer time scales in a
majority, under every d0. The difference is small: the median time scale of the first 10 PCs is 0.1
s under (a) and 0.125 s under (b), against 0.1 s for the last 10, at the resolution of the 100 ms
bins.

## fig4d-cauchy-model-power-law, fig4d-gaussian-model-no-power-law (blocked)

Simulations of a binary threshold network. The Methods give theta = 1, connection probability 0.01
and the critical point g = 1, but not the network size, the scaling of the weight distributions
with g (the formula is lost from the text), the run length, the initial state or inputs, or how
model activity is binned for PCA. Not run: `methods_underspecified`.

## Summary

| Claim | Outcome |
|---|---|
| dishes-analysed | reproduced (assumed: a day with no recording does not break a run) |
| activity-onset | blocked, metadata_insufficient |
| fig1b-async-to-bursts | reproduced (assumed d0 = 2) |
| fig1d-power-law-emerges | blocked, methods_underspecified |
| fig1e-example-dish | blocked, metadata_insufficient |
| fig1f-transition-15div | blocked, metadata_insufficient |
| fig1f-exponent-1p2 | blocked, metadata_insufficient (depends on the transition) |
| fig1d-shuffle | blocked, methods_underspecified |
| fig2b-spatial-structure | not reproduced (assumed M2, d0 = 2) |
| fig2c-locality-spectrum | reproduced (assumed M2 with L1, d0 = 2) |
| fig2d-locality-slope | partially reproduced (assumed M2 with L1) |
| fig2d-shuffle-locality | partially reproduced (assumed d0 = 2) |
| fig3c-avalanches-emerge | blocked, metadata_insufficient |
| fig3d-decay-rates-correlated | not reproduced |
| fig3ef-groups-cooccur | blocked, methods_underspecified |
| figs1-classified-by-avalanches | blocked, methods_underspecified |
| figs2-shuffle-breaks-cooccurrence | blocked, methods_underspecified |
| no-temporal-structure | not reproduced (assumed d0 = 2) |
| fig4d-cauchy-model-power-law | blocked, methods_underspecified |
| fig4d-gaussian-model-no-power-law | blocked, methods_underspecified |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** DIV are taken as days since a dish's first
recording plus 2, an assumption; the paper's DIV are not in the dataset. The fitted function, the
local Moran's I and the locality slope are readings of formulas lost from the text read; the
fitted b here (rank as x) is far below the paper's 0.1 boundary in every active recording, under
every fit reading and fit range tried, so the paper's b is probably computed differently. Bins
start at each file's first spike, as the files give no recording start. Spikes on readout channels
with no electrode (1.2e-4 of all) are left out.

**What was found about the dataset.** The files hold no days in vitro and no plating date (they
say so), so no result can be placed on the paper's DIV axis. The time of each spike is given from
the first detected spike, not from the recording start. Seven dishes and 129 recordings are
present, with some days missing; 17 recordings have a mean rate at or below 0.03 Hz, several of
them at the end of a dish's series (23282, 23302) or in its middle (23317, days 3 and 4; 23312,
day 22). Six recordings of dish 23311 (days 8 to 13) hold 338 to 702 electrodes, and one of 23280 (day 6)
holds 525, where the others hold 956 to 1012. No
recording is deposited twice: population counts of dishes recorded at the same times correlate at
most 0.225.